# Preprocesamiento de datos — Capa Bronze

Análisis exploratorio sobre los datos crudos ingestados en `nessie.bronze` para
justificar las reglas de limpieza aplicadas en silver. Se cubren cuatro bloques por
tabla: tipos de datos, datos faltantes, outliers y valores inválidos. Una última
sección resume cada regla silver con la evidencia cuantitativa encontrada en bronze.

Tablas analizadas: `nessie.bronze.yellow_taxi` y `nessie.bronze.taxi_zone_lookup`.

In [1]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("preprocess-bronze")
nessie.set_spark_branch(spark, "bronze")
spark.version

26/05/10 18:52:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/10 18:52:50 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/10 18:52:51 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to events/app-20260510185249-0007.inprogress. This is unsupported


'3.5.8'

## 1. Yellow Taxi

In [2]:
yt = spark.table("nessie.bronze.yellow_taxi").cache()
total_yt = yt.count()
print(f"Total de registros: {total_yt:,}")
print(f"Columnas: {len(yt.columns)}")

Total de registros: 67,978,426
Columnas: 20


### 1.1 Análisis de tipos de datos

La ingesta bronze unifica el *schema drift* del origen NYC TLC: entre años los
parquet originales alternan `INT`/`BIGINT` en los IDs, `INT`/`DOUBLE` en
`passenger_count` y `RatecodeID`, y renombran `Airport_fee` → `airport_fee` en 2025.
Bronze los normaliza a un esquema canónico con *casts* explícitos; aquí verificamos
el resultado.

In [3]:
# Esquema canonical resultante tras la ingesta
yt.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [4]:
# Histórico de evolución del esquema (Iceberg)
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-06 21:41:11.133|845165542525037693 |NULL               |true               |
|2026-05-06 21:41:55.311|482409955902079149 |845165542525037693 |true               |
|2026-05-06 21:42:43.407|1476967127453127779|482409955902079149 |true               |
|2026-05-06 21:43:20.301|2615922407221018522|1476967127453127779|true               |
|2026-05-06 21:43:56.69 |8060938301093377292|2615922407221018522|true               |
|2026-05-06 21:44:16.302|4572782734591690320|8060938301093377292|true               |
|2026-05-06 21:44:35.383|3192130432672798833|4572782734591690320|true               |
|2026-05-06 21:44:55.947|7523241285818114612|3192130432672798833|true               |
|2026-05-06 21:45:21.227|3866823855784470783|752324128

In [5]:
# cbd_congestion_fee (añadida en 2025) solo debería tener valores desde ese año.
# Lo validamos contando no-nulos por año.
from pyspark.sql.functions import col, year as spark_year, sum as spark_sum, when

yt.groupBy(spark_year(col("tpep_pickup_datetime")).alias("year")) \
  .agg(
      spark_sum(when(col("cbd_congestion_fee").isNotNull(), 1).otherwise(0)).alias("cbd_no_null"),
      spark_sum(when(col("airport_fee").isNotNull(), 1).otherwise(0)).alias("airport_fee_no_null"),
      spark_sum(when(col("congestion_surcharge").isNotNull(), 1).otherwise(0)).alias("congestion_no_null"),
  ) \
  .orderBy("year") \
  .show(50)

+----+-----------+-------------------+------------------+
|year|cbd_no_null|airport_fee_no_null|congestion_no_null|
+----+-----------+-------------------+------------------+
|2001|          0|                  3|                 3|
|2002|          0|                  2|                 3|
|2003|          0|                  5|                 6|
|2007|          1|                  1|                 1|
|2008|          0|                 19|                75|
|2009|          1|                 21|               183|
|2012|          0|                  1|                 1|
|2014|          0|                  1|                 1|
|2018|          0|                  0|                 0|
|2019|          0|                  0|          17664567|
|2020|          0|                  0|          15559767|
|2021|          0|                 24|                27|
|2022|          0|            8780168|           8780168|
|2023|          0|            9148254|           9148254|
|2024|        

### 1.2 Análisis de datos faltantes

Distinguimos dos tipos de nulos:
- **Estructurales**: columnas que no existían en determinados años (p.ej.
  `congestion_surcharge` desde 2019, `cbd_congestion_fee` desde 2025). No son
  errores y no se filtran en silver.
- **Anómalos**: nulos en columnas clave (`tpep_pickup_datetime`,
  `tpep_dropoff_datetime`) que impiden cualquier análisis temporal. Silver los
  elimina.

In [6]:
# Nulos totales por columna
null_counts = yt.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in yt.columns
])
null_counts.show(vertical=True)

-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 3034875  
 trip_distance         | 0        
 RatecodeID            | 3034875  
 store_and_fwd_flag    | 3034875  
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 7891090  
 airport_fee           | 41115652 
 cbd_congestion_fee    | 56780400 



In [7]:
# Porcentaje de nulos (sólo columnas con al menos un nulo)
row = null_counts.collect()[0].asDict()
pct = [(c, n, round(100 * n / total_yt, 2)) for c, n in row.items() if n > 0]
pct.sort(key=lambda t: -t[1])
for c, n, p in pct:
    print(f"  {c:<25s}  {n:>10,}  ({p:5.2f}%)")

  cbd_congestion_fee         56,780,400  (83.53%)
  airport_fee                41,115,652  (60.48%)
  congestion_surcharge        7,891,090  (11.61%)
  passenger_count             3,034,875  ( 4.46%)
  RatecodeID                  3,034,875  ( 4.46%)
  store_and_fwd_flag          3,034,875  ( 4.46%)


In [8]:
# Nulos por año en columnas con nulos estructurales vs anómalos.
# Confirma que congestion_surcharge y cbd_congestion_fee son estructurales
# (nulos concentrados en años anteriores a su introducción).
nulls_year = yt.groupBy(spark_year(col("tpep_pickup_datetime")).alias("year")).agg(
    spark_sum(col("passenger_count").isNull().cast("int")).alias("passenger_count_null"),
    spark_sum(col("RatecodeID").isNull().cast("int")).alias("RatecodeID_null"),
    spark_sum(col("store_and_fwd_flag").isNull().cast("int")).alias("store_and_fwd_null"),
    spark_sum(col("congestion_surcharge").isNull().cast("int")).alias("congestion_null"),
    spark_sum(col("airport_fee").isNull().cast("int")).alias("airport_fee_null"),
    spark_sum(col("cbd_congestion_fee").isNull().cast("int")).alias("cbd_null"),
).orderBy("year")
nulls_year.show(50)

+----+--------------------+---------------+------------------+---------------+----------------+--------+
|year|passenger_count_null|RatecodeID_null|store_and_fwd_null|congestion_null|airport_fee_null|cbd_null|
+----+--------------------+---------------+------------------+---------------+----------------+--------+
|2001|                   0|              0|                 0|              1|               1|       4|
|2002|                   0|              0|                 0|              0|               1|       3|
|2003|                   0|              0|                 0|              2|               3|       8|
|2007|                   0|              0|                 0|              0|               0|       0|
|2008|                   0|              0|                 0|             17|              73|      92|
|2009|                   0|              0|                 0|             44|             206|     226|
|2012|                   0|              0|            

In [9]:
# Filas con timestamps ausentes — no recuperables.
# Reutilizamos null_counts ya calculado arriba (evita dos escaneos completos).
print(f"tpep_pickup_datetime NULL:  {row['tpep_pickup_datetime']:,}")
print(f"tpep_dropoff_datetime NULL: {row['tpep_dropoff_datetime']:,}")

tpep_pickup_datetime NULL:  0
tpep_dropoff_datetime NULL: 0


### 1.3 Análisis de outliers y valores inválidos

Cada apartado muestra la evidencia detrás de una regla de limpieza silver.

#### 1.3.1 Timestamps fuera de rango

El dataset contiene viajes con fechas imposibles (años 2001, 2038, 2088…). El
recuento mensual en bronze ya los exhibe: provienen de errores en el taxímetro.

In [10]:
# Distribución de años fuera del rango esperado (2019-2025 de las cargas Q1)
from pyspark.sql.functions import month as spark_month

year_counts = yt.groupBy(
    spark_year(col("tpep_pickup_datetime")).alias("year")
).count().orderBy("year")
year_counts.show(50)

+----+--------+
|year|   count|
+----+--------+
|2001|       4|
|2002|       3|
|2003|       8|
|2007|       1|
|2008|      92|
|2009|     227|
|2012|       1|
|2014|       1|
|2018|     366|
|2019|22612159|
|2020|15711850|
|2021|      27|
|2022| 9071223|
|2023| 9384433|
|2024|      21|
|2025|11198003|
|2038|       4|
|2041|       1|
|2088|       2|
+----+--------+



#### 1.3.2 Timestamps inconsistentes (pickup ≥ dropoff)

Viajes cuyo *dropoff* es anterior o igual al *pickup*: imposibles físicamente.

In [11]:
bad_ts = yt.filter(col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime")).count()
print(f"Filas con pickup >= dropoff: {bad_ts:,}  ({100 * bad_ts / total_yt:.3f}%)")

Filas con pickup >= dropoff: 71,452  (0.105%)


#### 1.3.3 Importes y distancias negativos

Valores negativos en `trip_distance`, `fare_amount`, `tip_amount`, `tolls_amount`
y `total_amount` no tienen interpretación física en un viaje normal (en el feed
TLC suelen ser ajustes contables y *chargebacks*).

In [12]:
# Una sola pasada con agg() en vez de 5 filter().count() independientes.
neg_checks = [
    ("trip_distance", col("trip_distance") < 0),
    ("fare_amount",   col("fare_amount") < 0),
    ("total_amount",  col("total_amount") < 0),
    ("tip_amount",    col("tip_amount") < 0),
    ("tolls_amount",  col("tolls_amount") < 0),
]
neg_counts = yt.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(name)
    for name, cond in neg_checks
]).collect()[0].asDict()
for name, _ in neg_checks:
    n = neg_counts[name]
    print(f"  {name:<15s}  negativos: {n:>8,}  ({100 * n / total_yt:.3f}%)")

  trip_distance    negativos:    2,338  (0.003%)
  fare_amount      negativos:  741,889  (1.091%)
  total_amount     negativos:  394,318  (0.580%)
  tip_amount       negativos:    2,793  (0.004%)
  tolls_amount     negativos:   21,903  (0.032%)


In [13]:
# Resumen estadístico: min/max muestran la magnitud del ruido
yt.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "tolls_amount"
).summary("min", "25%", "50%", "75%", "max").show()

+-------+-------------+-----------+----------+------------+------------+
|summary|trip_distance|fare_amount|tip_amount|total_amount|tolls_amount|
+-------+-------------+-----------+----------+------------+------------+
|    min|       -30.62|    -1807.6|   -493.22|    -1832.85|     -142.17|
|    25%|          1.0|        7.0|       0.0|       11.76|         0.0|
|    50%|         1.68|       10.0|       2.0|       15.95|         0.0|
|    75%|          3.1|       16.0|       3.2|       22.89|         0.0|
|    max|    348798.53|   943274.8| 141492.02|  1084772.17|      3288.0|
+-------+-------------+-----------+----------+------------+------------+



#### 1.3.4 Outliers en importes y distancia

Importes y distancias correctamente firmados pero con valores extremos: los
percentiles altos (p95–p99.9) frente al máximo muestran la longitud de la cola
derecha, y los Top 20 por `fare_amount` y `trip_distance` permiten decidir si
esa cola es ruido del taxímetro o viajes legítimos caros (negociados con
`RatecodeID = 5`, trayectos al aeropuerto).

In [14]:
# Percentiles altos: revelan la forma de la cola derecha. Si p99 y max están
# muy separados hay una cola larga de valores atípicos candidata a ser recortada.
yt.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "tolls_amount"
).summary("50%", "90%", "95%", "99%", "99.9%", "max").show()

+-------+-------------+-----------+----------+------------+------------+
|summary|trip_distance|fare_amount|tip_amount|total_amount|tolls_amount|
+-------+-------------+-----------+----------+------------+------------+
|    50%|         1.68|       10.0|       2.0|       15.95|         0.0|
|    90%|         7.25|       28.9|       5.1|        38.1|         0.0|
|    95%|        11.45|       44.2|       8.0|       58.56|        5.76|
|    99%|        19.34|       70.0|      15.1|       91.74|        6.94|
|  99.9%|         27.7|      105.9|     23.96|       136.9|        17.5|
|    max|    348798.53|   943274.8| 141492.02|  1084772.17|      3288.0|
+-------+-------------+-----------+----------+------------+------------+



In [15]:
# Top 20 por fare_amount — ¿son negociados (RatecodeID=5), viajes al
# aeropuerto o basura? Revisar visualmente decide si el tope superior
# captura ruido o recorta viajes legítimos caros.
yt.select(
    "tpep_pickup_datetime", "trip_distance", "fare_amount", "total_amount",
    "RatecodeID", "payment_type", "PULocationID", "DOLocationID"
).orderBy(col("fare_amount").desc()).limit(20).show(truncate=False)

+--------------------+-------------+-----------+------------+----------+------------+------------+------------+
|tpep_pickup_datetime|trip_distance|fare_amount|total_amount|RatecodeID|payment_type|PULocationID|DOLocationID|
+--------------------+-------------+-----------+------------+----------+------------+------------+------------+
|2019-03-15 19:00:45 |0.0          |943274.8   |1084772.17  |1.0       |3           |233         |79          |
|2025-01-20 12:07:18 |1.6          |863372.12  |863380.37   |1.0       |4           |138         |8           |
|2019-02-25 18:43:23 |10.6         |671123.14  |671124.94   |1.0       |3           |138         |181         |
|2020-03-04 17:09:26 |2.1          |671100.14  |671103.17   |1.0       |4           |166         |143         |
|2019-01-11 19:33:15 |2.4          |623259.86  |623261.66   |1.0       |3           |237         |90          |
|2022-01-07 11:36:43 |3.3          |401092.32  |401095.62   |1.0       |4           |107         |140   

In [16]:
# Top 20 por trip_distance — un viaje dentro de NYC rara vez supera ~40 mi
# (JFK → norte de Westchester). Valores de cientos de millas son errores del
# taxímetro; deberían caer fuera de cualquier cota superior razonable.
yt.select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
    "fare_amount", "RatecodeID", "PULocationID", "DOLocationID"
).orderBy(col("trip_distance").desc()).limit(20).show(truncate=False)

+--------------------+---------------------+-------------+-----------+----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|RatecodeID|PULocationID|DOLocationID|
+--------------------+---------------------+-------------+-----------+----------+------------+------------+
|2022-02-15 18:24:00 |2022-02-15 18:37:00  |348798.53    |20.6       |NULL      |13          |107         |
|2023-02-21 05:19:00 |2023-02-21 05:36:00  |335004.33    |17.75      |NULL      |142         |42          |
|2022-02-02 07:34:00 |2022-02-02 07:53:00  |323919.67    |17.72      |NULL      |75          |48          |
|2025-03-27 20:52:00 |2025-03-27 20:59:00  |320136.29    |-4.75      |NULL      |161         |163         |
|2022-02-27 18:57:00 |2022-02-27 19:08:00  |314364.45    |9.96       |NULL      |125         |107         |
|2022-01-11 11:48:00 |2022-01-11 12:00:00  |306159.28    |14.96      |NULL      |116         |238         |
|2022-03-14 13:52:00 |2022-0

#### 1.3.5 IDs de localización fuera de rango

`PULocationID` y `DOLocationID` deben referenciar una zona válida (1–265 según
la tabla de referencia TLC). Los IDs 264 ("Unknown") y 265 ("N/A") se conservan
intencionadamente en silver — descartamos solo lo que está fuera del rango.

In [17]:
pu_out = yt.filter(~col("PULocationID").between(1, 265)).count()
do_out = yt.filter(~col("DOLocationID").between(1, 265)).count()
print(f"PULocationID fuera de [1,265]: {pu_out:,}")
print(f"DOLocationID fuera de [1,265]: {do_out:,}")

PULocationID fuera de [1,265]: 0
DOLocationID fuera de [1,265]: 0


#### 1.3.6 Códigos categóricos inválidos

Valores que no figuran en el diccionario de datos TLC 2024 indican
corrupción del taxímetro o registros de prueba.

In [18]:
# Valores presentes vs. valores válidos según diccionario TLC
print("VendorID (válidos 1,2,6,7):")
yt.groupBy("VendorID").count().orderBy("VendorID").show(truncate=False)

print("RatecodeID (válidos 1-6, 99):")
yt.groupBy("RatecodeID").count().orderBy("RatecodeID").show(truncate=False)

print("payment_type (válidos 0-6):")
yt.groupBy("payment_type").count().orderBy("payment_type").show(truncate=False)

print("store_and_fwd_flag (válidos Y, N):")
yt.groupBy("store_and_fwd_flag").count().orderBy("store_and_fwd_flag").show(truncate=False)

VendorID (válidos 1,2,6,7):


+--------+--------+
|VendorID|count   |
+--------+--------+
|1       |21265022|
|2       |46490296|
|4       |170407  |
|5       |307     |
|6       |25287   |
|7       |27107   |
+--------+--------+

RatecodeID (válidos 1-6, 99):


+----------+--------+
|RatecodeID|count   |
+----------+--------+
|NULL      |3034875 |
|1.0       |62319736|
|2.0       |1769052 |
|3.0       |144613  |
|4.0       |73250   |
|5.0       |432612  |
|6.0       |320     |
|99.0      |203968  |
+----------+--------+

payment_type (válidos 0-6):


+------------+--------+
|payment_type|count   |
+------------+--------+
|0           |3034875 |
|1           |49856393|
|2           |14259138|
|3           |359832  |
|4           |468152  |
|5           |36      |
+------------+--------+

store_and_fwd_flag (válidos Y, N):


+------------------+--------+
|store_and_fwd_flag|count   |
+------------------+--------+
|NULL              |3034875 |
|N                 |64400669|
|Y                 |542882  |
+------------------+--------+



In [19]:
# Recuento agregado de filas con al menos un código inválido — una sola pasada.
invalid_checks = [
    ("VendorID inválido",            ~col("VendorID").isin([1, 2, 6, 7])),
    ("RatecodeID inválido",          ~col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99])),
    ("payment_type inválido",        ~col("payment_type").isin([0, 1, 2, 3, 4, 5, 6])),
    ("store_and_fwd_flag inválido",  ~col("store_and_fwd_flag").isin(["Y", "N"])),
]
invalid_counts = yt.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(name)
    for name, cond in invalid_checks
]).collect()[0].asDict()
for name, _ in invalid_checks:
    print(f"{name:<32s} {invalid_counts[name]:,}")

VendorID inválido                170,714
RatecodeID inválido              0
payment_type inválido            0
store_and_fwd_flag inválido      0


#### 1.3.7 Passenger count fuera de lo físicamente posible

La ley NYC TLC permite 5 pasajeros en un taxi estándar, más la excepción de
menores de 7 años en el regazo de un adulto → techo razonable = 6. Los valores
7, 8 y 9 que aparecen en bronze son probables *sentinels* o errores del
taxímetro (no caben físicamente en un sedán de 4–5 plazas) y se descartan.

In [20]:
# Distribución completa (incluye nulos y valores extremos)
yt.groupBy("passenger_count").count().orderBy("passenger_count").show(50, truncate=False)

+---------------+--------+
|passenger_count|count   |
+---------------+--------+
|NULL           |3034875 |
|0.0            |1092491 |
|1.0            |47776997|
|2.0            |9454250 |
|3.0            |2440418 |
|4.0            |1170000 |
|5.0            |1871621 |
|6.0            |1137252 |
|7.0            |216     |
|8.0            |198     |
|9.0            |108     |
+---------------+--------+



In [21]:
# Filas fuera del rango [1,6] (incluye 0, 7, 8, 9 y nulos)
out_range = yt.filter(~col("passenger_count").between(1, 6)).count()
zero = yt.filter(col("passenger_count") == 0).count()
print(f"passenger_count fuera de [1,6] o nulo: {out_range:,}  ({100 * out_range / total_yt:.2f}%)")
print(f"  de los cuales passenger_count = 0:   {zero:,}")

passenger_count fuera de [1,6] o nulo: 1,093,013  (1.61%)
  de los cuales passenger_count = 0:   1,092,491


#### 1.3.8 Duplicados exactos

Silver aplica `dropDuplicates()` global como red de seguridad (mismas 20
columnas idénticas). Cuantificamos la magnitud aquí.

In [22]:
distinct_yt = yt.dropDuplicates().count()
dups = total_yt - distinct_yt
print(f"Filas totales:      {total_yt:,}")
print(f"Filas distintas:    {distinct_yt:,}")
print(f"Duplicados exactos: {dups:,}  ({100 * dups / total_yt:.3f}%)")

Filas totales:      67,978,426
Filas distintas:    67,965,432
Duplicados exactos: 12,994  (0.019%)


## 2. Taxi Zone Lookup

In [23]:
zl = spark.table("nessie.bronze.taxi_zone_lookup").cache()
total_zl = zl.count()
print(f"Total de registros: {total_zl}")
print(f"Columnas: {len(zl.columns)}")

Total de registros: 265
Columnas: 4


### 2.1 Análisis de tipos de datos

La tabla de referencia llega en CSV y sus tipos son estables. Silver renombra
las columnas a *snake_case* (`LocationID` → `location_id`, `Borough` →
`borough`, …) para homogeneizar el estilo del lakehouse.

In [24]:
zl.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



### 2.2 Análisis de datos faltantes

Toda zona debería tener `LocationID`, `Borough` y `Zone`. `service_zone` sí
puede ser nulo en alguna zona administrativa.

In [25]:
# Nulos por columna
null_counts_zl = zl.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in zl.columns
])
null_counts_zl.show()

+----------+-------+----+------------+
|LocationID|Borough|Zone|service_zone|
+----------+-------+----+------------+
|         0|      0|   0|           0|
+----------+-------+----+------------+



In [26]:
# Strings vacíos o solo whitespace (no detectados como nulos pero igual de problemáticos)
from pyspark.sql.functions import trim, length

empty_borough = zl.filter((length(trim(col("Borough"))) == 0) | col("Borough").isNull()).count()
empty_zone = zl.filter((length(trim(col("Zone"))) == 0) | col("Zone").isNull()).count()
empty_service = zl.filter((length(trim(col("service_zone"))) == 0) | col("service_zone").isNull()).count()
print(f"Borough vacío o nulo:      {empty_borough}")
print(f"Zone vacío o nulo:         {empty_zone}")
print(f"service_zone vacío o nulo: {empty_service}")

Borough vacío o nulo:      0
Zone vacío o nulo:         0
service_zone vacío o nulo: 0


### 2.3 Análisis de outliers y valores inválidos

In [27]:
# LocationID fuera del rango esperado [1, 265]
out_range_zl = zl.filter(~col("LocationID").between(1, 265)).count()
print(f"LocationID fuera de [1,265]: {out_range_zl}")

# Duplicados en la clave
dup_ids = zl.groupBy("LocationID").count().filter(col("count") > 1).count()
print(f"LocationID duplicados:       {dup_ids}")

LocationID fuera de [1,265]: 0
LocationID duplicados:       0


In [28]:
# Valores sentinel "Unknown" / "N/A" — se conservan (264, 265 son casos válidos del TLC)
zl.filter(
    col("Borough").isin(["Unknown", "N/A"]) | col("Zone").isin(["Unknown", "N/A"])
).show(truncate=False)

+----------+-------+--------------+------------+
|LocationID|Borough|Zone          |service_zone|
+----------+-------+--------------+------------+
|264       |Unknown|N/A           |N/A         |
|265       |N/A    |Outside of NYC|N/A         |
+----------+-------+--------------+------------+



In [29]:
# Dominio de boroughs (detecta typos o valores inesperados)
zl.groupBy("Borough").count().orderBy(col("count").desc()).show(truncate=False)

+-------------+-----+
|Borough      |count|
+-------------+-----+
|Manhattan    |69   |
|Queens       |69   |
|Brooklyn     |61   |
|Bronx        |43   |
|Staten Island|20   |
|Unknown      |1    |
|EWR          |1    |
|N/A          |1    |
+-------------+-----+



## 3. Resumen — reglas silver justificadas

Se recogen en una sola tabla las reglas aplicadas en
`src/pipelines/silver/clean_yellow_taxi.py` y `clean_zone_lookup.py`, junto al
recuento de filas bronze que cada regla descarta (evidencia cuantitativa).

In [30]:
# Cada regla aislada contra bronze — útil para el informe final del TFM.
# Una sola pasada con agg() en vez de 15 escaneos secuenciales.
# Nota: las reglas se aplican en cadena en silver; aquí se miden por separado
# (la suma no es la resta total porque una fila puede violar varias).
rules = [
    ("pickup NULL",                col("tpep_pickup_datetime").isNull()),
    ("dropoff NULL",               col("tpep_dropoff_datetime").isNull()),
    ("pickup >= dropoff",          col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime")),
    ("trip_distance < 0",          col("trip_distance") < 0),
    ("fare_amount < 0",            col("fare_amount") < 0),
    ("total_amount < 0",           col("total_amount") < 0),
    ("tip_amount < 0",             col("tip_amount") < 0),
    ("tolls_amount < 0",           col("tolls_amount") < 0),
    ("PULocationID fuera [1,265]", ~col("PULocationID").between(1, 265)),
    ("DOLocationID fuera [1,265]", ~col("DOLocationID").between(1, 265)),
    ("VendorID inválido",          ~col("VendorID").isin([1, 2, 6, 7])),
    ("RatecodeID inválido",        ~col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99])),
    ("payment_type inválido",      ~col("payment_type").isin([0, 1, 2, 3, 4, 5, 6])),
    ("store_and_fwd_flag inválido", ~col("store_and_fwd_flag").isin(["Y", "N"])),
    ("passenger_count fuera [1,6]", ~col("passenger_count").between(1, 6)),
]

rule_counts = yt.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(name)
    for name, cond in rules
]).collect()[0].asDict()

print(f"{'regla':<32s} {'filas':>12s}  {'%':>6s}")
print("-" * 55)
for name, _ in rules:
    n = rule_counts[name]
    print(f"{name:<32s} {n:>12,}  {100 * n / total_yt:>5.2f}%")

regla                                   filas       %
-------------------------------------------------------
pickup NULL                                 0   0.00%
dropoff NULL                                0   0.00%
pickup >= dropoff                      71,452   0.11%
trip_distance < 0                       2,338   0.00%
fare_amount < 0                       741,889   1.09%
total_amount < 0                      394,318   0.58%
tip_amount < 0                          2,793   0.00%
tolls_amount < 0                       21,903   0.03%
PULocationID fuera [1,265]                  0   0.00%
DOLocationID fuera [1,265]                  0   0.00%
VendorID inválido                     170,714   0.25%
RatecodeID inválido                         0   0.00%
payment_type inválido                       0   0.00%
store_and_fwd_flag inválido                 0   0.00%
passenger_count fuera [1,6]         1,093,013   1.61%


In [31]:
# Liberar caches
yt.unpersist()
zl.unpersist()
spark.stop()